# 3LC Low-Light Detection - Production Pipeline (YOLOv11n @640)

Competition: https://www.kaggle.com/competitions/the-3lc-low-light-detection-challenge

Model is FIXED: YOLOv11n from COCO yolo11n.pt at 640. Leaderboard = data quality, not architecture.

## Why this beats vanilla baselines

- Vanilla `yolo train` defaults (hsv_v=0.4, erasing=0.4) delete tiny Cups in dark frames. This uses low-light recipe hsv_v=0.7, erasing=0.2, mixup=0.1.
- Single conf=0.25 lets People (5966 boxes) drown Bus (565). This uses per-class conf: People 0.15, rare Bus/Cat/Dog 0.20-0.30.
- Trains once on noisy labels. This mines missing labels: conf>0.8 + IoU<0.1, human review, fix, retrain.
- Single-scale infer. This uses TTA multiscale 640+768 + flip (same weights, contract-safe).
- Strict sample_submission.csv schema check (715 rows).

Run top-to-bottom on Kaggle (T4 x2) or Colab (T4/A100). GPU required.

In [ ]:
# 0 - Setup: install, GPU check
!pip install -q ultralytics==8.3.169 opencv-python pyyaml pandas matplotlib seaborn 2>&1 | tail -2
import torch
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    !nvidia-smi --query-gpu=name,memory.total --format=csv 2>&1 | head -5
else:
    print('NO GPU - enable GPU: Runtime > Change runtime type > T4 GPU')

In [ ]:
# 1 - Config (single source of truth)
from pathlib import Path
WORK = Path('/kaggle/working') if Path('/kaggle').exists() else Path('/content')
KIT_ROOT = WORK / 'starter_kit'
DATA_YAML = WORK / 'dataset_abs.yaml'
CDN = 'https://competitions.3lc.ai/kaggle/exdark-low-light/starter-kit/v3'
CLASS_NAMES = {0:'Bicycle',1:'Boat',2:'Bottle',3:'Bus',4:'Car',5:'Cat',6:'Chair',7:'Cup',8:'Dog',9:'Motorbike',10:'People',11:'Table'}
PER_CLASS_CONF = {0:0.20,1:0.25,2:0.20,3:0.30,4:0.25,5:0.20,6:0.25,7:0.20,8:0.20,9:0.25,10:0.15,11:0.25}
LOWLIGHT_AUG = dict(hsv_h=0.015, hsv_s=0.7, hsv_v=0.7, fliplr=0.5, flipud=0.1, mosaic=1.0, mixup=0.1, copy_paste=0.1, degrees=5.0, translate=0.1, scale=0.5, shear=2.0, perspective=0.0, cutmix=0.0, erasing=0.2, auto_augment='randaugment', close_mosaic=10)
print('WORK:', WORK)
print('CDN:', CDN)

In [ ]:
# 2 - Download starter kit from 3LC CDN (625MB, resumable, sha256-verified)
import urllib.request, json, hashlib, zipfile
def sha256_file(p):
    import hashlib
    h = hashlib.sha256()
    with open(p,'rb') as f:
        for c in iter(lambda: f.read(1<<20), b''): h.update(c)
    return h.hexdigest()
manifest = json.load(urllib.request.urlopen(CDN + '/manifest.json'))
print('kit', manifest['kit_version'], manifest['file_count'], 'files, %.2f GB' % (manifest['total_bytes']/1e9))
dl_dir = WORK / 'kit_dl'; dl_dir.mkdir(exist_ok=True)
for a in manifest['archives']:
    name, size, sha = a['name'], a['bytes'], a['sha256']
    final, part = dl_dir/name, dl_dir/(name+'.part')
    if final.exists() and final.stat().st_size==size and sha256_file(final)==sha:
        print('SKIP', name); continue
    print('GET', name, '%.0fMB' % (size/1e6))
    start = part.stat().st_size if part.exists() else 0
    req = urllib.request.Request(CDN+'/'+name, headers={'Range':'bytes=%d-'%start} if start else {})
    with urllib.request.urlopen(req, timeout=60) as r, open(part, 'ab' if start else 'wb') as f:
        if start and getattr(r,'status',200)!=206:
            print('range ignored - restarting'); f.seek(0); f.truncate()
        while True:
            ch = r.read(1<<20)
            if not ch: break
            f.write(ch)
    assert sha256_file(part)==sha, 'hash fail '+name
    part.rename(final); print('OK', name)
print('Extracting...')
for a in manifest['archives']:
    with zipfile.ZipFile(dl_dir/a['name']) as z: z.extractall(WORK)
print('train/val/test:', len(list((KIT_ROOT/'images/train').glob('*.jpg'))), len(list((KIT_ROOT/'images/val').glob('*.jpg'))), len(list((KIT_ROOT/'images/test').glob('*.jpg'))))

In [ ]:
# 3 - Absolute dataset.yaml (fixes ultralytics relative-path bug)
import yaml
cfg = yaml.safe_load(open(KIT_ROOT/'dataset.yaml'))
cfg['path'] = str(KIT_ROOT)
cfg['train'] = str(KIT_ROOT/'images/train')
cfg['val'] = str(KIT_ROOT/'images/val')
cfg['test'] = str(KIT_ROOT/'images/test')
yaml.safe_dump(cfg, open(DATA_YAML,'w'))
print(open(DATA_YAML).read())
print('OK -', sum(1 for _ in (KIT_ROOT/'images/train').glob('*.jpg')), 'train')

In [ ]:
# 4 - EDA: class imbalance + darkness + box sizes
from collections import Counter
import cv2, numpy as np, random
import matplotlib.pyplot as plt
def count(split):
    c, tot = Counter(), 0
    for t in (KIT_ROOT/('labels/'+split)).glob('*.txt'):
        s = t.read_text().strip()
        if s:
            for ln in s.splitlines():
                if ln.strip(): c[int(float(ln.split()[0]))]+=1; tot+=1
    return c, tot
for split in ('train','val'):
    c, tot = count(split)
    print(split, tot, 'boxes')
    for k in sorted(c): print(' %2d %-8s %5d' % (k, CLASS_NAMES[k], c[k]))
random.seed(0)
imgs = random.sample(sorted((KIT_ROOT/'images/train').glob('*.jpg')), 200)
vs = [float(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2HSV)[:,:,2].mean()) for p in imgs]
print('brightness V mean=%.0f p10=%.0f (255=white; <60 near-dark)' % (np.mean(vs), np.percentile(vs,10)))
areas = []
for t in random.sample(sorted((KIT_ROOT/'labels/train').glob('*.txt')), 1000):
    for ln in t.read_text().strip().splitlines():
        p = ln.split()
        if len(p)>=5: areas.append(float(p[3])*float(p[4]))
areas = np.array(areas)
print('tiny<0.1%%: %.1f%% small<1%%: %.1f%% median=%.4f' % ((areas<0.001).mean()*100, (areas<0.01).mean()*100, np.median(areas)))
print('Takeaway: People dominates -> per-class conf. Tiny Cups -> erasing=0.2 + TTA. Dark -> hsv_v=0.7.')

In [ ]:
# 5 - Visual: raw vs CLAHE (what reviewers use to SEE missing boxes)
import cv2
import matplotlib.pyplot as plt
def clahe_img(p):
    img = cv2.imread(str(p)); lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l,a,b = cv2.split(lab)
    l = cv2.createCLAHE(2.0,(8,8)).apply(l)
    return cv2.cvtColor(cv2.merge((l,a,b)), cv2.COLOR_LAB2BGR)[:,:,::-1]
sample = sorted((KIT_ROOT/'images/train').glob('*.jpg'))[1]
raw = cv2.imread(str(sample))[:,:,::-1]
fig, ax = plt.subplots(1,2,figsize=(12,5))
ax[0].imshow(raw); ax[0].set_title('raw: '+sample.name); ax[0].axis('off')
ax[1].imshow(clahe_img(sample)); ax[1].set_title('CLAHE (review only)'); ax[1].axis('off')
plt.show()
print('GT:', open(KIT_ROOT/('labels/train/'+sample.stem+'.txt')).read().strip().splitlines()[:5])

In [ ]:
# 6 - BASELINE TRAIN: YOLOv11n @640, low-light recipe
from ultralytics import YOLO
EPOCHS, BATCH = 20, 16
model = YOLO('yolo11n.pt')
model.train(data=str(DATA_YAML), epochs=EPOCHS, imgsz=640, batch=BATCH, workers=2, device=0, amp=True, patience=25, save_period=5, exist_ok=True, name='prod_v1', plots=True, val=True, cls=0.7, cos_lr=True, seed=0, deterministic=True, **LOWLIGHT_AUG)
print('best: runs/detect/prod_v1/weights/best.pt')
# Reference: val mAP50 ~0.53 at ep2, ~0.70 at ep10.

In [ ]:
# 7 - Val score of baseline (your beat-this number)
from ultralytics import YOLO
m = YOLO('runs/detect/prod_v1/weights/best.pt')
print(m.val(data=str(DATA_YAML), imgsz=640))

In [ ]:
# 8 - MISSING-LABEL MINER: conf>0.8 + maxIoU<0.1 -> review queue
import json, csv
from ultralytics import YOLO
def iou(a,b):
    f = lambda bx: (bx[0]-bx[2]/2, bx[1]-bx[3]/2, bx[0]+bx[2]/2, bx[1]+bx[3]/2)
    x1,y1,x2,y2=f(a); xa,ya,xb,yb=f(b)
    iw,ih=max(0,min(x2,xb)-max(x1,xa)),max(0,min(y2,yb)-max(y1,ya))
    i=iw*ih; u=(x2-x1)*(y2-y1)+(xb-xa)*(yb-ya)-i
    return i/u if u>0 else 0.0
W='runs/detect/prod_v1/weights/best.pt'
CONF, IOU_T = 0.8, 0.1
model = YOLO(W)
queue=[]
for ip in sorted((KIT_ROOT/'images/train').glob('*.jpg')):
    lp = KIT_ROOT/('labels/train/'+ip.stem+'.txt')
    gt=[]
    if lp.exists() and (s:=lp.read_text().strip()):
        for ln in s.splitlines():
            if ln.strip(): t=ln.split(); gt.append((int(float(t[0])),tuple(map(float,t[1:5]))))
    r = model.predict(str(ip), conf=CONF, iou=0.5, verbose=False)[0]
    if r.boxes is None: continue
    for b in r.boxes:
        cls,cf=int(b.cls.item()),float(b.conf.item())
        try: xc,yc,w,h=b.xywhn.tolist()[0]
        except: x1,y1,x2,y2=b.xyxyn.tolist()[0]; xc,yc,w,h=(x1+x2)/2,(y1+y2)/2,x2-x1,y2-y1
        mi=max((iou((xc,yc,w,h),g) for _,g in gt), default=0.0)
        if mi<IOU_T:
            queue.append({'image':ip.name,'cls':cls,'cls_name':CLASS_NAMES[cls],'conf':round(cf,4),'box':[xc,yc,w,h],'max_iou':round(mi,4),'gt_count':len(gt),'status':'pending'})
queue.sort(key=lambda q:-q['conf'])
json.dump(queue, open(WORK/'review_queue.json','w'), indent=2)
with open(WORK/'review_queue.csv','w',newline='') as f:
    w=csv.writer(f); w.writerow(['image','cls','conf','xc','yc','w','h','max_iou','gt_count'])
    for q in queue: w.writerow([q['image'],q['cls'],q['conf'],*q['box'],q['max_iou'],q['gt_count']])
print('queued', len(queue), 'candidates across', len(set(q['image'] for q in queue)), 'images')
for q in queue[:10]: print(' ', q['image'], q['cls_name'], q['conf'], 'iou=', q['max_iou'])
print('RULES: inspect before accept. Check CLAHE images next, THEN apply.')

In [ ]:
# 9 - Review viz: top-6 candidates raw vs CLAHE (human gate - do NOT skip)
import matplotlib.patches as patches
q = json.load(open(WORK/'review_queue.json'))[:20]
print('reviewing', len(q))
for item in q[:6]:
    ip = KIT_ROOT/('images/train/'+item['image'])
    raw = cv2.imread(str(ip))[:,:,::-1]
    enh = clahe_img(ip)
    for img_arr, title in ((raw,'raw'),(enh,'CLAHE')):
        fig, ax = plt.subplots(figsize=(6,4)); ax.imshow(img_arr)
        H,Wd = cv2.imread(str(ip)).shape[:2]
        xc,yc,w,h = item['box']
        ax.add_patch(patches.Rectangle(((xc-w/2)*Wd,(yc-h/2)*H),w*Wd,h*H,fill=False,edgecolor='lime',lw=2))
        ax.set_title(title+' '+item['image']+' pred='+item['cls_name']+' '+str(item['conf'])); ax.axis('off'); plt.show()

In [ ]:
# 10 - APPLY human-accepted fixes to dataset_fixed (only status == 'accepted')
# List the queue indices you confirmed in the CLAHE review (cell 9), e.g. [0, 2, 3].
# Bulk-accepting by confidence is banned by the Rules, so nothing is applied by default.
import shutil, yaml
from collections import defaultdict
from pathlib import Path as P
ACCEPTED = []
REJECTED = []
queue = json.load(open(WORK/'review_queue.json'))
for i in ACCEPTED: queue[i]['status'] = 'accepted'
for i in REJECTED: queue[i]['status'] = 'rejected'
json.dump(queue, open(WORK/'review_queue.json','w'), indent=2)
accepted = [x for x in queue if x['status']=='accepted']
assert accepted, 'No accepted candidates: review cell 9, fill ACCEPTED, re-run this cell'
print('accepting %d/%d reviewed candidates' % (len(accepted), len(queue)))
FIXED = WORK/'dataset_fixed'
if FIXED.exists(): shutil.rmtree(FIXED)
shutil.copytree(KIT_ROOT, FIXED, ignore=shutil.ignore_patterns('*.zip'))
by_img = defaultdict(list)
for x in accepted: by_img[x['image']].append(x)
for img, boxes in by_img.items():
    lp = FIXED/('labels/train/'+P(img).stem+'.txt')
    lines = lp.read_text().strip().splitlines() if lp.exists() and lp.read_text().strip() else []
    for b in boxes:
        xc,yc,w,h=b['box']; lines.append('%d %.6f %.6f %.6f %.6f' % (b['cls'],xc,yc,w,h))
    lp.write_text(chr(10).join(lines)+chr(10))
print('patched', len(by_img), 'images')
cfg=yaml.safe_load(open(DATA_YAML)); cfg['path']=str(FIXED); cfg['train']=str(FIXED/'images/train'); cfg['val']=str(FIXED/'images/val'); cfg['test']=str(FIXED/'images/test')
yaml.safe_dump(cfg, open(WORK/'dataset_fixed.yaml','w'))
print('wrote', WORK/'dataset_fixed.yaml')

In [ ]:
# 11 - RETRAIN on fixed labels
from ultralytics import YOLO
model = YOLO('yolo11n.pt')
model.train(data=str(WORK/'dataset_fixed.yaml'), epochs=20, imgsz=640, batch=16, workers=2, device=0, amp=True, patience=25, save_period=5, exist_ok=True, name='prod_v2_fixed', plots=True, val=True, cls=0.7, cos_lr=True, seed=0, deterministic=True, **LOWLIGHT_AUG)
print('best: runs/detect/prod_v2_fixed/weights/best.pt')
print(YOLO('runs/detect/prod_v2_fixed/weights/best.pt').val(data=str(WORK/'dataset_fixed.yaml'), imgsz=640))

In [ ]:
# 12 - SUBMISSION: TTA multiscale + per-class conf + strict validation
import csv
from ultralytics import YOLO
from pathlib import Path as PP
W='runs/detect/prod_v2_fixed/weights/best.pt'
if not PP(W).exists(): W='runs/detect/prod_v1/weights/best.pt'
model = YOLO(W); print('weights:', W)
order=[r['image_id'] for r in list(csv.DictReader(open(KIT_ROOT/'sample_submission.csv')))]
print('test images:', len(order))
rows=[]
for idx, iid in enumerate(order):
    ip = KIT_ROOT/('images/test/'+iid+'.jpg')
    preds=[]
    for sz in (640,768):
        for flip in (False,True):
            import cv2 as _cv
            img=_cv.imread(str(ip))
            if flip: img=_cv.flip(img,1)
            rs=model.predict(img, conf=0.15, iou=0.45, imgsz=sz, verbose=False)[0]
            if rs.boxes is None: continue
            for b in rs.boxes:
                cls,cf=int(b.cls.item()),float(b.conf.item())
                if cf < PER_CLASS_CONF.get(cls,0.25): continue
                try: xc,yc,w,h=b.xywhn.tolist()[0]
                except: x1,y1,x2,y2=b.xyxyn.tolist()[0]; xc,yc,w,h=(x1+x2)/2,(y1+y2)/2,x2-x1,y2-y1
                if flip: xc=1.0-xc
                preds.append((cf,'%d %.4f %.6f %.6f %.6f %.6f' % (cls,cf,xc,yc,w,h)))
    preds=sorted(preds, reverse=True)[:100]
    rows.append((idx,iid,' '.join(p for _,p in preds) if preds else 'no box'))
    if idx%150==0: print(' %d/%d %s: %d boxes' % (idx,len(order),iid,len(preds)))
with open(WORK/'submission.csv','w',newline='') as f:
    w=csv.writer(f); w.writerow(['id','image_id','prediction_string']); w.writerows(rows)
got=list(csv.DictReader(open(WORK/'submission.csv')))
assert len(got)==715 and set(got[0].keys())=={'id','image_id','prediction_string'}
for r in got:
    if r['prediction_string']!='no box': assert len(r['prediction_string'].split())%6==0
print('OK submission.csv: %d rows, %d with boxes' % (len(got), sum(1 for r in got if r['prediction_string']!='no box')))
print(got[:2])

In [ ]:
# 13 - Submit (needs kaggle.json: Kaggle Settings > API > Create New Token)
# from google.colab import files; files.upload()
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle competitions submit -c the-3lc-low-light-detection-challenge -f /kaggle/working/submission.csv -m "yolo11n lowlight-aug + label-fix + TTA"
# !kaggle competitions submissions -c the-3lc-low-light-detection-challenge 2>&1 | head
print('Uncomment the 4 lines above after uploading kaggle.json. Daily limit: 3.')

## Same pipeline from the command line

The repo's `src/` modules run the same steps outside the notebook. Set `LOWLIGHT_KIT` if the kit is not in the 3LC plugin folder:

```
pip install -r requirements.txt
export LOWLIGHT_KIT=/kaggle/working/starter_kit
python -m src.train_production --epochs 20 --batch 16 --name prod_v1
python -m src.mine_missing --weights runs/detect/prod_v1/weights/best.pt --conf 0.8
# set status accepted/rejected in review_queue.json after visual review
python -m src.apply_fixes --queue review_queue.json --apply
python -m src.train_production --data dataset_fixed/dataset.yaml --name prod_v2_fixed
python -m src.predict --weights runs/detect/prod_v2_fixed/weights/best.pt
```

A FastAPI serving layer (`src/serve.py`) is planned and not in the repo yet.